[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stevenelliottjr/healthcare-analytics-course-data/blob/main/notebooks/primer_python_pandas_sql_answers.ipynb) · [Course home](https://colab.research.google.com/github/stevenelliottjr/healthcare-analytics-course-data/blob/main/notebooks/00_course_home.ipynb)

**Google Colab version.** These are the primer's answers, shared on purpose: try each exercise first, then
compare. Every self-check prints **Correct** here. To run them, **File → Save a copy in Drive** (or just
**Runtime → Run all** in playground mode; nothing you change here is saved). Nothing to submit.

# Primer — Python, pandas, and SQL for this course (answers)
**This copy has every exercise filled in.** Use it when you are stuck, or to compare after you finish. There is usually more than one right answer: if your version prints Correct, it is right.

*Self-paced, before week 1. Practice only: nothing here is graded. About 3 to 4 hours in all.*

This primer is for you if you have never written Python, or wrote some a long time ago. It teaches the
small set of tools the labs use, on the same hospital data the labs use. You do not need to memorize
anything. The goal is simple: in week 1, the code in the labs should look familiar.

Do it after *Lab 0 — Getting started in Colab* (the Colab tour). It leads straight into *Lab 1*: by the end you will be able to read every line of it.

**How it works.** Six parts. Each part has a short explanation, a worked example you just run, and three
to five exercises. After each exercise there is a **self-check** cell. Run it. It prints **Correct**, or
**Not yet** with a hint about what to change. In exercise cells, replace each `None` (or the line marked
`# TODO`) with your code. You never need to edit a self-check cell.

**Sittings.** Plan on four sittings of 45 to 60 minutes:

| Sitting | Parts | Time |
|---|---|---|
| 1 | Part 1. Python basics | about 45 min |
| 2 | Part 2. pandas I: looking at a table | about 35 min |
| 3 | Part 3. pandas II: numbers, missing values, groups, joins | about 45 min |
| 4 | Parts 4 to 6. SQL, charts, ready check | about 80 min |

**Coming back later.** A notebook forgets everything when its kernel restarts. When you return, click
**Run all** (it takes under a minute), then scroll to where you stopped. Unfinished exercises will just
say "Not yet"; nothing breaks.

**Stuck?** Give any exercise about fifteen minutes. Then look at the answers notebook, or post in the
course channel. Reading an answer and then typing it yourself is a perfectly good way to learn. You may
ask an AI assistant to explain a line, but try writing the code yourself first: the point of the primer
is to get your hands used to it.

The first code cell is the **course setup cell**. Every lab starts with it. You will read it line by line
in week 3; for now, just run it. The second cell sets up the self-checks.

In [ ]:
# --- Course setup: run this cell first -------------------------------------------
# Loaders for the public datasets this course uses. The canonical copy of this code
# is labs/course_data.py in the course repo; you will read it line by line in week 3.
import io, json, urllib.request, urllib.parse
from pathlib import Path
import pandas as pd
import duckdb

CACHE = Path("data"); CACHE.mkdir(exist_ok=True)

def sql(query: str) -> pd.DataFrame:
    """Run SQL over any DataFrame in this notebook by its variable name (DuckDB)."""
    return duckdb.sql(query).df()

def _get(url, params=None, timeout=120) -> bytes:
    if params:
        url = url + "?" + urllib.parse.urlencode(params)
    req = urllib.request.Request(url, headers={"User-Agent": "healthcare-analytics-course/1.0"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

# ---- CMS Care Compare (data.cms.gov/provider-data) --------------------------------
# File names change at every CMS refresh; dataset identifiers do not. We ask the
# catalog for the current CSV URL every time instead of hard-coding a file name.
CMS_CATALOG = "https://data.cms.gov/provider-data/api/1/metastore/schemas/dataset/items"
CMS_IDS = {
    "hospital_general_info": "xubh-q36u",   # one row per hospital
    "hrrp":                  "9n3s-kdb3",   # Hospital Readmissions Reduction Program, one row per hospital x measure
    "unplanned_visits":      "632h-zaca",   # readmissions / ED visits, one row per hospital x measure
    "complications_deaths":  "ynj2-r877",
    "timely_effective":      "yv7e-xc69",
}

def cms_csv_url(identifier: str) -> str:
    # Ask for the one dataset by id; the full keyword search can take minutes or time out.
    item = json.loads(_get(f"{CMS_CATALOG}/{identifier}", timeout=30))
    for d in item.get("distribution", []):
        if d.get("mediaType") == "text/csv":
            return d["downloadURL"]
    raise KeyError(f"No CSV distribution found for CMS dataset {identifier}")

# Course mirror: a snapshot of every CMS file the labs use, for when CMS is down or rate-limits
# a classroom downloading the same file at once. The live source is always tried first.
COURSE_DATA_BASE = "https://raw.githubusercontent.com/stevenelliottjr/healthcare-analytics-course-data/main/data"

def _fetch_or_mirror(path: Path, fetch, mirror_file: str) -> None:
    """Save `path` from the live source; if that fails, from the course mirror."""
    try:
        path.write_bytes(fetch())
    except Exception as e:
        print(f"Live download failed ({type(e).__name__}); using the course mirror for {path.name}.")
        path.write_bytes(_get(f"{COURSE_DATA_BASE}/{mirror_file}", timeout=300))

def load_cms(name: str, **read_csv_kwargs) -> pd.DataFrame:
    """Download (once) and read a Care Compare file. Everything is read as TEXT on purpose:
    IDs keep their leading zeros and 'Not Available' stays visible instead of silently
    becoming NaN. Converting columns to numbers is your job, and part of the lesson."""
    path = CACHE / f"cms_{name}.csv"
    if not path.exists():
        _fetch_or_mirror(path, lambda: _get(cms_csv_url(CMS_IDS[name]), timeout=300), f"cms/cms_{name}.csv")
    kw = dict(dtype=str, keep_default_na=False, encoding="utf-8-sig")
    kw.update(read_csv_kwargs)
    return pd.read_csv(path, **kw)

def to_num(s: pd.Series) -> pd.Series:
    """Text -> number; anything that is not a number ('Not Available', 'N/A', '') becomes NaN."""
    return pd.to_numeric(s, errors="coerce")

# ---- CDC PLACES, county release (Socrata API on data.cdc.gov) ---------------------
PLACES_COUNTY = "https://data.cdc.gov/resource/swc5-untb.json"
PLACES_SELECT = ("year,stateabbr,statedesc,locationname,locationid,categoryid,measureid,measure,"
                 "data_value,low_confidence_limit,high_confidence_limit,totalpopulation")

def load_places(measure_ids, data_value_type="Age-adjusted prevalence", state=None, page=50000) -> pd.DataFrame:
    """County-level model-based prevalence estimates (percent of adults). locationid is the
    5-digit county FIPS. Note: Connecticut reports *planning regions*, not counties, since 2022."""
    where = "measureid in(" + ",".join(f"'{m}'" for m in measure_ids) + f") AND data_value_type='{data_value_type}'"
    if state:
        where += f" AND stateabbr='{state}'"
    frames, offset = [], 0
    while True:
        chunk = pd.DataFrame(json.loads(_get(PLACES_COUNTY, {"$where": where, "$select": PLACES_SELECT,
                                                               "$limit": page, "$offset": offset})))
        frames.append(chunk)
        if len(chunk) < page:
            break
        offset += page
    df = pd.concat(frames, ignore_index=True)
    for c in ["data_value", "low_confidence_limit", "high_confidence_limit", "totalpopulation"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

# ---- Week 4 powerups (links checked 2026-10-26; 2022 is the latest SVI release) ----
SVI_COUNTY_URL = "https://svi.cdc.gov/Documents/Data/2022/csv/states_counties/SVI_2022_US_county.csv"
def load_svi_county() -> pd.DataFrame:
    """CDC/ATSDR Social Vulnerability Index 2022, one row per county.
    Key columns: FIPS, STATE, COUNTY, E_TOTPOP, RPL_THEMES (overall percentile 0-1),
    RPL_THEME1..RPL_THEME4. The value -999 means 'not available'."""
    path = CACHE / "svi_2022_county.csv"
    if not path.exists():
        path.write_bytes(_get(SVI_COUNTY_URL, timeout=300))
    return pd.read_csv(path, dtype={"FIPS": str}).replace(-999, pd.NA)

CHR_URL = "https://www.countyhealthrankings.org/sites/default/files/media/document/analytic_data2025_v3.csv"
def load_county_health_rankings() -> pd.DataFrame:
    """County Health Rankings analytic file. Row 0 holds readable labels and row 1 the variable
    codes; we skip the labels so the codes (fipscode, v001..v1xx) become the column names.
    The data dictionary on the CHR site maps each v-code to a measure."""
    path = CACHE / "chr_analytic_2025_v3.csv"
    if not path.exists():
        path.write_bytes(_get(CHR_URL, timeout=300))
    return pd.read_csv(path, skiprows=[0], dtype={"fipscode": str, "statecode": str, "countycode": str})

print("course setup ready")


In [ ]:
# --- Primer helpers: run this cell too ---------------------------------------------
# self_check() runs the checks after each exercise. You do not need to read or edit this cell.
import traceback
import numpy as np
import matplotlib.pyplot as plt

RESULTS = {}   # exercise label -> True (correct) / False (not yet)
NOT_NONE = "Replace None with your answer, then run the exercise cell before this one."

def self_check(label, steps):
    """steps: a list of (test, hint). The first test that fails prints its hint; never raises."""
    for test, hint in steps:
        try:
            ok = bool(test())
            err = None
        except Exception as e:
            ok, err = False, f"{type(e).__name__}: {e}"
        if not ok:
            if callable(hint):
                try:
                    hint = hint()
                except Exception:
                    hint = "Check your answer against the instructions."
            if err:
                hint += f"  (Python said: {err})"
            RESULTS[label] = False
            print(f"{label}: Not yet. {hint}")
            return False
    RESULTS[label] = True
    print(f"{label}: Correct")
    return True

def close(a, b, tol=1e-6):
    """True when two numbers are equal up to rounding."""
    return a is not None and abs(float(a) - float(b)) <= tol

def keyed(result, key):
    """Index an answer by `key`, whether it came from SQL (key is a column) or pandas (key is the index)."""
    if isinstance(result, pd.DataFrame) and key in result.columns:
        return result.set_index(key)
    return result

def counts_of(result, key):
    """A count answer (Series, or a DataFrame whose last column is the count) as a plain dict."""
    r = keyed(result, key)
    if isinstance(r, pd.DataFrame):
        r = r.iloc[:, -1]
    return {k: int(v) for k, v in r.items()}

print("self-checks ready")


## Part 1. Python basics (about 45 minutes)
Python is a calculator that remembers. Eight ideas cover almost everything the labs need.

- **Values and types.** `4` is a whole number (`int`), `3.5` is a decimal (`float`), `"CT"` is text (`str`),
  and `True` / `False` are yes-or-no values (`bool`). `type(x)` tells you which. Text always goes in quotes.
  `"4"` and `4` are different things: one is text, one is a number. That difference matters a lot in this course.
- **Variables.** `n_beds = 120` stores 120 under the name `n_beds`. Here `=` means "store", not "equals".
  Names use lowercase letters and underscores, no spaces.
- **Lists.** `[4, 3, 5]` is an ordered collection. `stars[0]` is the *first* item, because Python counts
  from 0. `len()`, `max()`, `min()` and `sum()` work on lists.
- **Dictionaries.** `{"name": "Harbor Clinic", "state": "CT"}` maps *keys* to *values*.
  `clinic["state"]` looks a value up. Keys are case-sensitive: `"state"` is not `"State"`.
- **Loops.** `for s in stars:` runs the indented lines once for each item. The indentation (four spaces)
  is how Python knows which lines belong to the loop.
- **if.** `if s >= 4:` runs the indented lines only when the condition is true; `else:` covers the rest.
  `==` compares; `=` stores.
- **Functions.** `def` packages lines you want to reuse. `return` hands back the result.
- **f-strings.** `f"{name} has {n_beds} beds"` puts values into text. Note the `f` before the quote.

A `#` starts a comment: Python ignores the rest of the line. The worked example below uses every idea once.
Run it and match each printed line to the code that made it.

In [ ]:
# Values and types (all numbers in Part 1 are made up)
n_beds = 120
occupancy = 0.8
home_state = "CT"
is_teaching = True
print(type(n_beds), type(occupancy), type(home_state), type(is_teaching))

# "4" (text) is not 4 (a number). int() and float() turn text into numbers.
print("4" + "4", "versus", 4 + 4, "versus", int("4") + int("4"))

# Lists: ordered, counted from 0
stars = [4, 3, 5, 2]
print("first:", stars[0], " last:", stars[-1], " how many:", len(stars), " best:", max(stars))

# Dictionaries: look values up by key
hospital = {"name": "Elm City Hospital", "state": "CT", "rating": 3}
print(hospital["name"], "is in", hospital["state"])
hospital["beds"] = 154               # add a new key and value

# A loop with an if
for s in stars:
    if s >= 4:
        print(s, "stars: high")
    else:
        print(s, "stars: not high")

# A function, used inside an f-string
def percent(part, whole):
    # part / whole as a percent, rounded to one decimal place
    return round(100 * part / whole, 1)

print(f"{hospital['name']} has {hospital['beds']} beds and is {percent(96, 120)}% full.")


In [ ]:
# Exercise 1.1 — lists
# Six hospitals in a made-up county have these star ratings:
county_stars = [3, 4, 2, 5, 4, 3]

n_county = len(county_stars)
top_county = max(county_stars)
third_county = county_stars[2]


In [ ]:
# Self-check: just run this cell.
self_check("Exercise 1.1", [
    (lambda: all(x is not None for x in (n_county, top_county, third_county)), NOT_NONE),
    (lambda: n_county == 6, "n_county should count the items: try len(county_stars)."),
    (lambda: top_county == 5, "top_county should be the largest rating: try max(county_stars)."),
    (lambda: third_county != county_stars[3], "county_stars[3] is the FOURTH item. Counting from 0, the third is at position 2."),
    (lambda: third_county == 2, "third_county should be county_stars[2]."),
])


In [ ]:
# Exercise 1.2 — dictionaries
clinic = {"name": "Harbor Clinic", "state": "CT", "visits_2025": 18250}

clinic_state = clinic["state"]
clinic["visits_2026"] = 19400
clinic_growth = clinic["visits_2026"] - clinic["visits_2025"]


In [ ]:
# Self-check: just run this cell.
self_check("Exercise 1.2", [
    (lambda: clinic_state is not None, NOT_NONE),
    (lambda: clinic_state == "CT", "clinic_state should be clinic[\"state\"], which is the text \"CT\"."),
    (lambda: "visits_2026" in clinic, "clinic has no key \"visits_2026\" yet. Write clinic[\"visits_2026\"] = 19400 (keys are case-sensitive)."),
    (lambda: clinic["visits_2026"] == 19400, "clinic[\"visits_2026\"] should be the NUMBER 19400, without quotes."),
    (lambda: clinic_growth is not None, "Now set clinic_growth."),
    (lambda: clinic_growth == 1150, "clinic_growth should be clinic[\"visits_2026\"] - clinic[\"visits_2025\"], which is 1150."),
])


In [ ]:
# Exercise 1.3 — a loop with an if
# This is how star ratings arrive in the course data: as TEXT, with words for missing values.
raw_ratings = ["4", "3", "Not Available", "5", "Not Available", "2"]

n_missing = 0
total_stars = 0
for r in raw_ratings:
    if r == "Not Available":
        n_missing = n_missing + 1
    else:
        total_stars = total_stars + int(r)


In [ ]:
# Self-check: just run this cell.
self_check("Exercise 1.3", [
    (lambda: not (n_missing == 0 and total_stars == 0), "Both are still 0. Write the loop: for r in raw_ratings: (then indent the next lines)."),
    (lambda: isinstance(total_stars, (int, float)), "total_stars is not a number. Did you add text? Use int(r) to turn \"4\" into 4."),
    (lambda: n_missing == 2, lambda: f"n_missing is {n_missing}; there are 2 \"Not Available\" values. Compare with == \"Not Available\", spelled exactly."),
    (lambda: total_stars == 14, lambda: f"total_stars is {total_stars}; 4 + 3 + 5 + 2 = 14. Add int(r) only when r is not \"Not Available\"."),
])


In [ ]:
# Exercise 1.4 — a function and an f-string
def describe(name, n_stars):
    return f"{name}: {n_stars} stars"

describe("Harbor Clinic", 4)


In [ ]:
# Self-check: just run this cell.
self_check("Exercise 1.4", [
    (lambda: describe("Harbor Clinic", 4) is not None, "The function still returns None. Replace None with an f-string, like f\"{name}: ...\"."),
    (lambda: isinstance(describe("Harbor Clinic", 4), str), "The function should return text (an f-string)."),
    (lambda: describe("Harbor Clinic", 4) == "Harbor Clinic: 4 stars",
        lambda: f"It returned {describe('Harbor Clinic', 4)!r}. It should return 'Harbor Clinic: 4 stars' (check the colon, the spaces and 'stars')."),
    (lambda: describe("Elm City Hospital", 2) == "Elm City Hospital: 2 stars",
        "It works for Harbor Clinic but not for other inputs. Use {name} and {n_stars} inside the f-string instead of typing the values."),
])


### Reading an error message
Errors are normal. Everyone who writes code sees several an hour. An error message (a *traceback*) looks
scary, but it is a set of directions, and you read it **from the bottom up**. The cell below makes a real
error on purpose: it tries to add up ratings that include the words `Not Available`.

In [ ]:
# A real error, on purpose. Normally an error stops the cell and prints the message in red.
# Here we catch it with try / except so the rest of the notebook keeps running.
# You will not need to write try / except yourself in this course.
def to_whole_number(text):
    return int(text)

demo_ratings = ["4", "3", "Not Available"]
try:
    demo_total = 0
    for item in demo_ratings:
        demo_total = demo_total + to_whole_number(item)
except Exception:
    print(traceback.format_exc())


How to read the message above:

1. **The last line first.** It names the kind of error and what went wrong:
   `ValueError: invalid literal for int() with base 10: 'Not Available'`. In plain words: `int()` was given the
   text `'Not Available'`, which is not a whole number.
2. **The line above it** shows the exact line that failed, `return int(text)`, inside the function `to_whole_number`.
3. **Keep going up** to see who called that function: the loop line `demo_total = demo_total + to_whole_number(item)`.
   The fix usually belongs in the highest line that *you* wrote. Here: skip `'Not Available'` before converting,
   as you did in Exercise 1.3.

Common errors in this course:

| Error | Usually means |
|---|---|
| `NameError: name 'x' is not defined` | A typo in the name, or the cell that creates `x` has not been run yet |
| `KeyError: 'State'` | No such key or column. Check the spelling, the capitals and the spaces |
| `TypeError` | Mixing types, such as `"4" + 4` (text plus a number) |
| `ValueError` | Right type, wrong value, such as `int("Not Available")` |
| `SyntaxError` / `IndentationError` | A missing quote, bracket, colon, or a line indented wrongly |

When you ask for help, paste the **whole** message, not only the last line.

In [ ]:
# Exercise 1.5 — read an error message
# Someone ran this code:
#
#     clinic = {"name": "Harbor Clinic", "state": "CT"}
#     print(clinic["State"])
#
# and got this:
#
#     Traceback (most recent call last):
#       File "<cell>", line 2, in <module>
#         print(clinic["State"])
#               ~~~~~~^^^^^^^^^
#     KeyError: 'State'
#
error_kind = "KeyError"
missing_key = "State"
working_key = "state"


In [ ]:
# Self-check: just run this cell.
self_check("Exercise 1.5", [
    (lambda: all(x is not None for x in (error_kind, missing_key, working_key)), NOT_NONE),
    (lambda: str(error_kind).strip().lower() == "keyerror", "The kind of error is the word before the colon on the last line."),
    (lambda: missing_key == "State", "missing_key is the text after the colon on the last line, exactly as written (capital S)."),
    (lambda: working_key == "state", "Look at the keys in the dictionary: which one matches, apart from the capital letter?"),
])


## Part 2. pandas I: looking at a table (about 35 minutes)
**pandas** is the Python library for tables. A table is a **DataFrame**: rows and named columns. A single
column is a **Series**. The course loads CMS *Hospital General Information* (one row per Medicare-certified
hospital, about 5,400 rows) into a DataFrame called `hospitals`.

- **Look.** `hospitals.head()` shows the first five rows. `hospitals.shape` is (rows, columns).
  `hospitals.columns` lists the column names. `hospitals.dtypes` gives each column's type; `object` means text.
- **Select columns.** `hospitals["State"]` is one column. `hospitals[["Facility Name", "State"]]` is a smaller
  table. Note the double brackets: the outer pair selects, the inner pair is a list of names.
- **Filter rows.** `hospitals[hospitals["State"] == "CT"]` keeps the Connecticut rows. The part inside,
  `hospitals["State"] == "CT"`, is a column of True / False, one per row; pandas keeps the True rows.
  Combine conditions with `&` (and) or `|` (or), and put each condition in its own parentheses.
- **Sort.** `.sort_values("Facility Name")` sorts A to Z; add `ascending=False` for largest first.
- **Count.** `.value_counts()` on a column counts each distinct value, most common first.

Column names must match exactly, spaces and capitals included: `"Hospital Type"`, not `"hospital type"`.
If you get a `KeyError`, print `hospitals.columns` and copy the name from there.

In [ ]:
hospitals = load_cms("hospital_general_info")   # downloads once, then reads from disk
print(hospitals.shape)                           # (rows, columns)
print(hospitals.dtypes.head())                   # 'object' means text
hospitals.head(3)


In [ ]:
# One column (a Series) and several columns (a DataFrame)
names = hospitals["Facility Name"]
where = hospitals[["Facility Name", "City/Town", "State"]]
print(type(names), type(where))

# Filter: hospitals in Rhode Island
is_ri = hospitals["State"] == "RI"         # True / False for every row
ri = hospitals[is_ri]
print(len(ri), "hospitals in Rhode Island")

# Two conditions: in Rhode Island AND no emergency department. Each condition in parentheses.
ri_no_er = hospitals[(hospitals["State"] == "RI") & (hospitals["Emergency Services"] == "No")]

# Sort, then show two columns
print(ri_no_er.sort_values("Facility Name")[["Facility Name", "Hospital Type"]])

# Count each distinct value. The last line of a cell is displayed automatically.
hospitals["Hospital Ownership"].value_counts().head()


In [ ]:
# Exercise 2.1 — the size of the table
n_hospitals = hospitals.shape[0]     # or len(hospitals)
n_columns = hospitals.shape[1]       # or len(hospitals.columns)


In [ ]:
# Self-check: just run this cell.
self_check("Exercise 2.1", [
    (lambda: all(x is not None for x in (n_hospitals, n_columns)), NOT_NONE),
    (lambda: not (n_hospitals == hospitals.shape[1] and n_columns == hospitals.shape[0]), "You have them swapped: .shape is (rows, columns)."),
    (lambda: n_hospitals == len(hospitals), "n_hospitals should be the number of rows: hospitals.shape[0] or len(hospitals)."),
    (lambda: n_columns == hospitals.shape[1], "n_columns should be hospitals.shape[1]."),
])


In [ ]:
# Exercise 2.2 — select columns
name_city_type = hospitals[["Facility Name", "City/Town", "Hospital Type"]]


In [ ]:
# Self-check: just run this cell.
self_check("Exercise 2.2", [
    (lambda: name_city_type is not None, NOT_NONE),
    (lambda: isinstance(name_city_type, pd.DataFrame), "That is one column (a Series). For a table, use double brackets: hospitals[[\"A\", \"B\", \"C\"]]."),
    (lambda: list(name_city_type.columns) == ["Facility Name", "City/Town", "Hospital Type"],
        lambda: f"Your columns are {list(name_city_type.columns)}. You want Facility Name, City/Town, Hospital Type, in that order."),
    (lambda: len(name_city_type) == len(hospitals), "Keep every row: select columns only, do not filter."),
])


In [ ]:
# Exercise 2.3 — filter rows
ct_hospitals = hospitals[hospitals["State"] == "CT"]


In [ ]:
# Self-check: just run this cell.
self_check("Exercise 2.3", [
    (lambda: ct_hospitals is not None, NOT_NONE),
    (lambda: isinstance(ct_hospitals, pd.DataFrame), "ct_hospitals should be a table of rows: hospitals[ <condition> ]."),
    (lambda: len(ct_hospitals) > 0, "No rows came back. Did you write \"Connecticut\"? The column uses \"CT\"."),
    (lambda: set(ct_hospitals["State"]) == {"CT"}, "Some rows are not in CT. The condition is hospitals[\"State\"] == \"CT\"."),
    (lambda: len(ct_hospitals) == (hospitals["State"] == "CT").sum(), "Some CT hospitals are missing. Filter on State only."),
])


In [ ]:
# Exercise 2.4 — two conditions, then sort
ma_cah = hospitals[(hospitals["State"] == "MA") &
                   (hospitals["Hospital Type"] == "Critical Access Hospitals")].sort_values("Facility Name")
ma_cah[["Facility Name", "City/Town"]]


In [ ]:
# Self-check: just run this cell.
self_check("Exercise 2.4", [
    (lambda: ma_cah is not None, NOT_NONE),
    (lambda: isinstance(ma_cah, pd.DataFrame) and len(ma_cah) > 0, "No rows. Check the spelling of \"MA\" and of \"Critical Access Hospitals\"."),
    (lambda: set(ma_cah["State"]) == {"MA"} and set(ma_cah["Hospital Type"]) == {"Critical Access Hospitals"},
        "Some rows fail one of the two conditions. Join them with &, each in parentheses: (cond1) & (cond2)."),
    (lambda: len(ma_cah) == ((hospitals["State"] == "MA") & (hospitals["Hospital Type"] == "Critical Access Hospitals")).sum(),
        "Some matching hospitals are missing. Use only the two conditions."),
    (lambda: ma_cah["Facility Name"].is_monotonic_increasing, "The rows are right but not sorted. Add .sort_values(\"Facility Name\")."),
])


In [ ]:
# Exercise 2.5 — count with value_counts
type_counts = hospitals["Hospital Type"].value_counts()
most_common_type = type_counts.index[0]      # value_counts puts the most common first
type_counts


In [ ]:
# Self-check: just run this cell.
self_check("Exercise 2.5", [
    (lambda: all(x is not None for x in (type_counts, most_common_type)), NOT_NONE),
    (lambda: isinstance(type_counts, pd.Series), "type_counts should be hospitals[\"Hospital Type\"].value_counts()."),
    (lambda: type_counts.sum() == len(hospitals), "The counts should add up to the number of hospitals. Count the Hospital Type column."),
    (lambda: isinstance(most_common_type, str), "most_common_type should be the type's name (text), not its count. Try type_counts.index[0]."),
    (lambda: most_common_type == hospitals["Hospital Type"].value_counts().index[0], "That is not the most common type. value_counts() lists the most common first."),
])


## Part 3. pandas II: numbers, missing values, groups, joins (about 45 minutes)
**Why everything arrives as text.** `load_cms` reads every column as text, on purpose. Two reasons:

- **IDs.** `Facility ID` values look like `070010`. Read as a number, that becomes 70010, and it no longer
  matches the same hospital in other CMS files.
- **Missing values.** The rating column holds 1 to 5 *or* the words `Not Available`. As text you can see the
  missing values and count them before deciding what to do.

So you convert the columns you need yourself, with `to_num(...)` from the setup cell. It turns numbers into
numbers and everything else (`Not Available`, blanks) into **`NaN`** ("not a number"), pandas' marker for missing.

**Missing is not zero.** `NaN` is left out of `.mean()`, `.sum()` and `.count()`. Replacing it with 0
(for example with `.fillna(0)`) treats a hospital with no rating as a zero-star hospital and drags every
average down. Never do that in this course. Useful tools: `.isna()`, `.notna()`, `.isna().sum()`, `.dropna()`.

**Groups.** `df.groupby("Hospital Type")` splits the table by value, and `.agg(...)` summarizes each group.
Name each result: `.agg(hospitals=("Facility ID", "count"), mean_rating=("rating", "mean"))`.
`"count"` counts the non-missing values in that column.

**Joins.** `pd.merge(left, right, on="Facility ID")` lines up rows that share a key. The default is an
**inner join**: rows with no match on the other side are **dropped, with no warning**. Always count rows
before and after a join. `how="left"` keeps every row of the left table; `indicator=True` adds a `_merge`
column saying whether each row matched (`both`) or not (`left_only`).

**New columns.** `df["new"] = ...` makes a column from other columns, one value per row.

In [ ]:
# Why IDs stay text: as a number, the leading zero disappears
print(int("070010"))

# Convert a count column: death-rate measures where the hospital did worse than the national rate.
# 'Not Available' becomes NaN, never 0.
hospitals["mort_worse"] = to_num(hospitals["Count of MORT Measures Worse"])
print(hospitals["mort_worse"].dtype)                        # float64: numbers now
print(hospitals["mort_worse"].isna().sum(), "hospitals have no value (NaN)")
print(hospitals["mort_worse"].value_counts(dropna=False).sort_index())   # dropna=False shows the NaN row too

# The star rating as a number. The rest of the primer uses this column.
hospitals["rating"] = to_num(hospitals["Hospital overall rating"])


In [ ]:
# groupby + agg: one row per Hospital Type. The outer parentheses let a long line wrap.
by_type = (hospitals
           .groupby("Hospital Type")
           .agg(hospitals=("Facility ID", "count"),          # every hospital
                with_mort_data=("mort_worse", "count"),      # only hospitals with a value
                mean_mort_worse=("mort_worse", "mean"))      # mean skips NaN
           .sort_values("hospitals", ascending=False))
by_type


In [ ]:
# A small hand-made table of bed counts for five Connecticut hospitals (the bed numbers are made up).
# The fifth ID has lost its leading zero, the way a spreadsheet does when it treats IDs as numbers.
ct_ids = hospitals.loc[hospitals["State"] == "CT", "Facility ID"].sort_values().head(5).tolist()
bed_table = pd.DataFrame({
    "Facility ID": ct_ids[:4] + [ct_ids[4].lstrip("0")],
    "beds": [120, 85, 310, 46, 200],
})
print(bed_table, "\n")

joined = pd.merge(bed_table, hospitals[["Facility ID", "Facility Name"]], on="Facility ID")   # inner join
print(len(bed_table), "rows in bed_table;", len(joined), "rows after the join. One hospital vanished, with no warning.")
joined


In [ ]:
# Exercise 3.1 — text to numbers
hospitals["readm_worse"] = to_num(hospitals["Count of READM Measures Worse"])
n_readm_missing = hospitals["readm_worse"].isna().sum()
n_readm_missing


In [ ]:
# Self-check: just run this cell.
self_check("Exercise 3.1", [
    (lambda: "readm_worse" in hospitals.columns, "There is no column hospitals[\"readm_worse\"] yet. Write hospitals[\"readm_worse\"] = to_num(...)."),
    (lambda: pd.api.types.is_numeric_dtype(hospitals["readm_worse"]), "readm_worse is still text. Wrap the column in to_num( )."),
    (lambda: hospitals["readm_worse"].isna().sum() > 0, "readm_worse has no missing values. Did you fill them with 0? Missing must stay NaN."),
    (lambda: n_readm_missing is not None, NOT_NONE),
    (lambda: n_readm_missing == to_num(hospitals["Count of READM Measures Worse"]).isna().sum(),
        "n_readm_missing should be hospitals[\"readm_worse\"].isna().sum()."),
])


In [ ]:
# Exercise 3.2 — missing is not zero
four = pd.Series(["4", "Not Available", "2", "3"])

four_num = to_num(four)
right_mean = four_num.mean()              # NaN is skipped: (4 + 2 + 3) / 3
wrong_mean = four_num.fillna(0).mean()    # (4 + 0 + 2 + 3) / 4: a hospital with no rating counted as 0 stars
print(right_mean, wrong_mean)


In [ ]:
# Self-check: just run this cell.
self_check("Exercise 3.2", [
    (lambda: all(x is not None for x in (right_mean, wrong_mean)), NOT_NONE),
    (lambda: close(right_mean, 3.0), lambda: f"right_mean is {right_mean}. It should be (4 + 2 + 3) / 3 = 3.0: to_num(four).mean()."),
    (lambda: close(wrong_mean, 2.25), lambda: f"wrong_mean is {wrong_mean}. It should be (4 + 0 + 2 + 3) / 4 = 2.25: to_num(four).fillna(0).mean()."),
])
print("The gap between the two is what filling missing values with 0 would do to every average in this course.")


In [ ]:
# Exercise 3.3 — groupby and agg
by_state = (hospitals
            .groupby("State")
            .agg(hospitals=("Facility ID", "count"),
                 rated=("rating", "count"),
                 mean_rating=("rating", "mean"))
            .sort_values("mean_rating", ascending=False))
by_state.head(10)


In [ ]:
# Self-check: just run this cell.
self_check("Exercise 3.3", [
    (lambda: by_state is not None, NOT_NONE),
    (lambda: isinstance(by_state, pd.DataFrame), "by_state should be a table: hospitals.groupby(\"State\").agg(...)."),
    (lambda: {"hospitals", "rated", "mean_rating"} <= set(by_state.columns),
        lambda: f"Your columns are {list(by_state.columns)}. Name them hospitals, rated, mean_rating inside .agg( )."),
    (lambda: keyed(by_state, "State").loc["CT", "hospitals"] == (hospitals["State"] == "CT").sum(),
        "The hospitals column is off. Group by \"State\" and count \"Facility ID\"."),
    (lambda: keyed(by_state, "State").loc["CT", "rated"] == hospitals.loc[hospitals["State"] == "CT", "rating"].notna().sum(),
        "rated should count only hospitals with a rating: (\"rating\", \"count\"). Counting \"Facility ID\" counts everyone."),
    (lambda: close(keyed(by_state, "State").loc["CT", "mean_rating"], hospitals.loc[hospitals["State"] == "CT", "rating"].mean()),
        "mean_rating is off. Use (\"rating\", \"mean\") on the numeric rating column from Part 3's first cell."),
    (lambda: by_state["mean_rating"].dropna().is_monotonic_decreasing, "Sort with .sort_values(\"mean_rating\", ascending=False)."),
])


In [ ]:
# Exercise 3.4 — find what the inner join dropped, then fix it
bed_left = pd.merge(bed_table, hospitals[["Facility ID", "Facility Name"]],
                    on="Facility ID", how="left", indicator=True)
n_unmatched = (bed_left["_merge"] == "left_only").sum()
print(bed_left, "\n")

bed_fixed = bed_table.copy()
bed_fixed["Facility ID"] = bed_fixed["Facility ID"].str.zfill(6)
fixed_join = pd.merge(bed_fixed, hospitals[["Facility ID", "Facility Name"]], on="Facility ID")
print(len(fixed_join), "rows after fixing the key")


In [ ]:
# Self-check: just run this cell.
self_check("Exercise 3.4", [
    (lambda: bed_left is not None, NOT_NONE),
    (lambda: len(bed_left) == len(bed_table), "A left join keeps every row of bed_table, so bed_left should have 5 rows. Use how=\"left\"."),
    (lambda: "_merge" in bed_left.columns, "bed_left has no _merge column. Add indicator=True to pd.merge."),
    (lambda: n_unmatched == 1, lambda: f"n_unmatched is {n_unmatched}; one ID has no match. Count (bed_left[\"_merge\"] == \"left_only\").sum()."),
    (lambda: bed_table["Facility ID"].str.len().min() == 5,
        "bed_table itself was changed. Make bed_fixed with bed_table.copy() and fix the copy. (Rerun the Part 3 join cell to restore bed_table.)"),
    (lambda: bed_fixed is not None and fixed_join is not None, "Now make bed_fixed and fixed_join (TODO 2)."),
    (lambda: bed_fixed["Facility ID"].str.len().eq(6).all(), "Every Facility ID in bed_fixed should have 6 characters: .str.zfill(6)."),
    (lambda: len(fixed_join) == 5, lambda: f"fixed_join has {len(fixed_join)} rows; with the key fixed, all 5 hospitals should match."),
])


In [ ]:
# Exercise 3.5 — new columns
fixed_join["large"] = fixed_join["beds"] >= 200
fixed_join["label"] = fixed_join["Facility Name"] + " (" + fixed_join["beds"].astype(str) + " beds)"
fixed_join


In [ ]:
# Self-check: just run this cell.
self_check("Exercise 3.5", [
    (lambda: fixed_join is not None, "Finish Exercise 3.4 first: fixed_join is still None."),
    (lambda: "large" in fixed_join.columns, "There is no \"large\" column yet: fixed_join[\"large\"] = fixed_join[\"beds\"] >= 200."),
    (lambda: fixed_join["large"].tolist() == (fixed_join["beds"] >= 200).tolist(), "\"large\" should be True exactly when beds >= 200 (200 counts as large)."),
    (lambda: "label" in fixed_join.columns, "There is no \"label\" column yet."),
    (lambda: fixed_join["label"].iloc[0] == f"{fixed_join['Facility Name'].iloc[0]} ({fixed_join['beds'].iloc[0]} beds)",
        lambda: f"Your first label is {fixed_join['label'].iloc[0]!r}. It should look like 'NAME (120 beds)': name, space, parenthesis, number, ' beds)'."),
])


## Part 4. SQL with DuckDB (about 40 minutes)
SQL is the other language for tables, and the one most healthcare data teams use every day. In this course
you write SQL with the `sql()` helper from the setup cell. `sql("SELECT ... FROM hospitals")` runs the query
on the pandas DataFrame named `hospitals` and gives back a new DataFrame. (We use `sql()` so the same code works in any notebook: Colab, Hex, or Jupyter on a laptop.)

A query is written in this order:

```sql
SELECT   the columns you want, and calculations
FROM     the table
WHERE    which rows to keep          (before grouping)
GROUP BY how to group
HAVING   which groups to keep        (after grouping)
ORDER BY how to sort                 (add DESC for largest first)
LIMIT    how many rows to return
```

- **Quoting.** Column names go in **double quotes**: `"Hospital Type"` (required when a name has spaces or
  symbols, so we quote them all). Text values go in **single quotes**: `WHERE "State" = 'CT'`.
  Mixing the two up is the most common SQL error in this course.
- **Triple quotes.** `sql('''...''')` lets a query run over several lines, which makes it readable.
- **Numbers.** In SQL, `TRY_CAST("Hospital overall rating" AS DOUBLE)` does what `to_num` does: a number,
  or `NULL` (SQL's missing) for `Not Available`. `COUNT(*)` counts rows, `COUNT(x)` counts non-missing values,
  and `AVG(x)` skips missing values.
- **Joins.** `FROM hospitals h JOIN regions r ON h."State" = r."State"` is an inner join, with the same
  warning as in pandas: unmatched rows disappear. `h` and `r` are short nicknames for the two tables.

Each exercise asks for the SQL first, then the same answer in pandas. Doing both is a habit for the term:
one checks the other.

In [ ]:
# SQL: the first five Rhode Island hospitals, A to Z
ri_sql = sql('''
    SELECT "Facility Name", "City/Town", "Hospital Type"
    FROM hospitals
    WHERE "State" = 'RI'
    ORDER BY "Facility Name"
    LIMIT 5
''')
print(ri_sql, "\n")

# pandas: the same thing
ri_pd = (hospitals[hospitals["State"] == "RI"]
         .sort_values("Facility Name")
         [["Facility Name", "City/Town", "Hospital Type"]]
         .head(5))
print(ri_pd)


In [ ]:
# GROUP BY and HAVING: hospital types with at least 100 hospitals, and how many of them have a rating
sql('''
    SELECT "Hospital Type",
           COUNT(*) AS hospitals,
           COUNT(TRY_CAST("Hospital overall rating" AS DOUBLE)) AS rated
    FROM hospitals
    GROUP BY "Hospital Type"
    HAVING COUNT(*) >= 100
    ORDER BY hospitals DESC
''')


In [ ]:
# Exercise 4.1 — SELECT, WHERE, ORDER BY, LIMIT
sql_ct_er = sql('''
    SELECT "Facility Name", "City/Town"
    FROM hospitals
    WHERE "State" = 'CT' AND "Emergency Services" = 'Yes'
    ORDER BY "City/Town", "Facility Name"
    LIMIT 10
''')

pd_ct_er = (hospitals[(hospitals["State"] == "CT") & (hospitals["Emergency Services"] == "Yes")]
            .sort_values(["City/Town", "Facility Name"])
            [["Facility Name", "City/Town"]]
            .head(10))
sql_ct_er


In [ ]:
# Self-check: just run this cell.
expected_41 = lambda: (hospitals[(hospitals["State"] == "CT") & (hospitals["Emergency Services"] == "Yes")]
                       .sort_values(["City/Town", "Facility Name"])["Facility Name"].head(10).tolist())
for label_41, answer_41 in [("Exercise 4.1 (SQL)", sql_ct_er), ("Exercise 4.1 (pandas)", pd_ct_er)]:
    self_check(label_41, [
        (lambda: answer_41 is not None, NOT_NONE),
        (lambda: "Facility Name" in answer_41.columns, "Include the \"Facility Name\" column."),
        (lambda: len(answer_41) == 10, lambda: f"You have {len(answer_41)} rows; you want the first 10 (LIMIT 10, or .head(10))."),
        (lambda: set(answer_41["Facility Name"]) <= set(hospitals.loc[(hospitals["State"] == "CT") & (hospitals["Emergency Services"] == "Yes"), "Facility Name"]),
            "Some rows are not CT hospitals with an emergency department. Check both conditions (AND in SQL, & in pandas)."),
        (lambda: answer_41["Facility Name"].tolist() == expected_41(), "Right hospitals, wrong order. Sort by City/Town first, then Facility Name."),
    ])


In [ ]:
# Exercise 4.2 — GROUP BY and HAVING
sql_big_states = sql('''
    SELECT "State", COUNT(*) AS hospitals
    FROM hospitals
    GROUP BY "State"
    HAVING COUNT(*) > 100
    ORDER BY hospitals DESC
''')

state_counts = hospitals["State"].value_counts()
pd_big_states = state_counts[state_counts > 100]
sql_big_states


In [ ]:
# Self-check: just run this cell.
expected_42 = lambda: {k: int(v) for k, v in hospitals["State"].value_counts().items() if v > 100}
for label_42, answer_42 in [("Exercise 4.2 (SQL)", sql_big_states), ("Exercise 4.2 (pandas)", pd_big_states)]:
    self_check(label_42, [
        (lambda: answer_42 is not None, NOT_NONE),
        (lambda: set(counts_of(answer_42, "State")) == set(expected_42()),
            lambda: f"You have {len(counts_of(answer_42, 'State'))} states; {len(expected_42())} states have more than 100 hospitals. Keep groups with COUNT(*) > 100 (HAVING), or counts > 100 in pandas."),
        (lambda: counts_of(answer_42, "State") == expected_42(), "Right states, wrong counts. Count all hospitals in each state: COUNT(*)."),
        (lambda: pd.Series(list(counts_of(answer_42, "State").values())).is_monotonic_decreasing, "Sort largest first: ORDER BY hospitals DESC."),
    ])


In [ ]:
# Exercise 4.3 — numbers in SQL
sql_er_rating = sql('''
    SELECT "Emergency Services",
           COUNT(*) AS hospitals,
           COUNT(TRY_CAST("Hospital overall rating" AS DOUBLE)) AS rated,
           AVG(TRY_CAST("Hospital overall rating" AS DOUBLE))   AS mean_rating
    FROM hospitals
    GROUP BY "Emergency Services"
    ORDER BY "Emergency Services"
''')

pd_er_rating = (hospitals.groupby("Emergency Services")
                .agg(hospitals=("Facility ID", "count"),
                     rated=("rating", "count"),
                     mean_rating=("rating", "mean")))
sql_er_rating


In [ ]:
# Self-check: just run this cell.
for label_43, answer_43 in [("Exercise 4.3 (SQL)", sql_er_rating), ("Exercise 4.3 (pandas)", pd_er_rating)]:
    self_check(label_43, [
        (lambda: answer_43 is not None, NOT_NONE),
        (lambda: {"hospitals", "rated", "mean_rating"} <= set(answer_43.columns),
            lambda: f"Your columns are {list(answer_43.columns)}. Name them hospitals, rated, mean_rating (AS ... in SQL)."),
        (lambda: keyed(answer_43, "Emergency Services").loc["Yes", "hospitals"] == (hospitals["Emergency Services"] == "Yes").sum(),
            "hospitals should count every row in the group: COUNT(*)."),
        (lambda: keyed(answer_43, "Emergency Services").loc["Yes", "rated"] == hospitals.loc[hospitals["Emergency Services"] == "Yes", "rating"].notna().sum(),
            "rated should count only real ratings: COUNT(TRY_CAST(...)) in SQL, (\"rating\", \"count\") in pandas."),
        (lambda: close(keyed(answer_43, "Emergency Services").loc["No", "mean_rating"],
                       hospitals.loc[hospitals["Emergency Services"] == "No", "rating"].mean()),
            "mean_rating is off. Average the converted rating; never turn 'Not Available' into 0."),
    ])


In [ ]:
# Exercise 4.4 — JOIN
regions = pd.DataFrame({
    "State":  ["CT", "MA", "ME", "NH", "RI", "VT", "NJ", "NY", "PA"],
    "region": ["New England"] * 6 + ["Mid-Atlantic"] * 3,
})

sql_regions = sql('''
    SELECT r.region, COUNT(*) AS hospitals
    FROM hospitals h
    JOIN regions r ON h."State" = r."State"
    GROUP BY r.region
    ORDER BY hospitals DESC
''')

hosp_regions = pd.merge(hospitals, regions, on="State")      # inner join
pd_regions = hosp_regions["region"].value_counts()
n_outside = len(hospitals) - len(hosp_regions)
print(n_outside, "hospitals were dropped by the inner join")
sql_regions


In [ ]:
# Self-check: just run this cell.
expected_44 = lambda: {k: int(v) for k, v in pd.merge(hospitals, regions, on="State")["region"].value_counts().items()}
for label_44, answer_44 in [("Exercise 4.4 (SQL)", sql_regions), ("Exercise 4.4 (pandas)", pd_regions)]:
    self_check(label_44, [
        (lambda: answer_44 is not None, NOT_NONE),
        (lambda: set(counts_of(answer_44, "region")) == {"New England", "Mid-Atlantic"},
            "You should have one row per region (New England, Mid-Atlantic). Group by the region column from the regions table."),
        (lambda: counts_of(answer_44, "region") == expected_44(),
            lambda: f"Your counts are {counts_of(answer_44, 'region')}; expected {expected_44()}. Join on State and count hospitals."),
    ])
self_check("Exercise 4.4 (dropped rows)", [
    (lambda: n_outside is not None, NOT_NONE),
    (lambda: n_outside == len(hospitals) - sum(expected_44().values()),
        "n_outside is the number of hospitals before the join minus the number after it."),
])


## Part 5. Charts (about 25 minutes)
The course draws charts with **matplotlib**. Every chart follows the same pattern:

```python
fig, ax = plt.subplots(figsize=(7, 4))      # a blank figure (fig) with one chart area (ax)
ax.bar(labels, values)                      # vertical bars; ax.barh(...) for horizontal; ax.plot(x, y) for a line
ax.set_title("What is shown, and where the data is from")
ax.set_xlabel("...")
ax.set_ylabel("... (units)")
plt.show()
```

Use **bars** to compare amounts across categories. Use a **line** for change over something ordered,
usually time. In this primer each chart gets its own names (`fig_51, ax_51`, ...) so each self-check can
find its chart.

**What makes a chart honest**
- **Bars start at zero.** The eye reads a bar's length as its value. Cut off the bottom and a small difference
  looks huge. (A line chart may zoom in, because the eye reads its slope, not its length.)
- **Label the units.** "3.2" of what? Stars, percent, hospitals, dollars per stay?
- **Say what is counted and where it is from**, in the title or a note.
- **Show how many.** A mean of 4.2 from 112 hospitals is not the same as a mean of 3.2 from 2,600.

The worked example draws the same numbers twice: once misleading, once honest.

In [ ]:
rated = hospitals.dropna(subset=["rating"])          # only hospitals with a rating
type_rating = rated.groupby("Hospital Type")["rating"].agg(["mean", "count"]).sort_values("mean")
labels = [f"{t} (n={n:,})" for t, n in zip(type_rating.index, type_rating["count"])]

fig_demo, (ax_bad, ax_good) = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)

ax_bad.barh(labels, type_rating["mean"])
ax_bad.set_xlim(3.0, 4.3)                          # the trick: the axis starts at 3, not 0
ax_bad.set_title("Misleading: the axis starts at 3")

ax_good.barh(labels, type_rating["mean"])
ax_good.set_xlim(0, 5)                             # bars start at zero
ax_good.set_xlabel("mean overall star rating (1 to 5 stars)")
ax_good.set_title("Honest: starts at 0, units labeled")

fig_demo.suptitle("Mean CMS overall star rating by hospital type, rated hospitals only")
plt.tight_layout()
plt.show()


In [ ]:
# Exercise 5.1 — a bar chart
new_england = ["CT", "MA", "ME", "NH", "RI", "VT"]
fig_51, ax_51 = plt.subplots(figsize=(7, 4))
ne_counts = hospitals.loc[hospitals["State"].isin(new_england), "State"].value_counts()
ax_51.bar(ne_counts.index, ne_counts.values)
ax_51.set_title("Medicare-certified hospitals in New England (CMS Hospital General Information)")
ax_51.set_ylabel("hospitals")
ax_51.set_xlabel("state")
plt.show()


In [ ]:
# Self-check: just run this cell.
self_check("Exercise 5.1", [
    (lambda: len(ax_51.patches) > 0, "No bars yet. Use ax_51.bar(labels, values) between subplots and plt.show()."),
    (lambda: len(ax_51.patches) == 6, lambda: f"You have {len(ax_51.patches)} bars; there are 6 New England states."),
    (lambda: sorted(round(p.get_height()) for p in ax_51.patches) ==
             sorted(hospitals.loc[hospitals["State"].isin(new_england), "State"].value_counts().tolist()),
        "The bar heights do not match the hospital counts. Count with value_counts() on the filtered State column."),
    (lambda: ax_51.get_ylim()[0] == 0, "The y axis does not start at 0. Bars must start at zero; remove any set_ylim."),
    (lambda: ax_51.get_title().strip() != "", "Add a title: ax_51.set_title(\"...\")."),
    (lambda: ax_51.get_ylabel().strip() != "", "Add a y-axis label with the unit: ax_51.set_ylabel(\"hospitals\")."),
])


In [ ]:
# Exercise 5.2 — a line chart
ed_visits = pd.DataFrame({
    "month":  ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"],
    "visits": [1320, 1280, 1150, 1010, 950, 900, 880, 910, 990, 1100, 1240, 1400],
})
fig_52, ax_52 = plt.subplots(figsize=(7, 4))
ax_52.plot(ed_visits["month"], ed_visits["visits"], marker="o")
ax_52.set_title("Emergency department visits by month, one clinic (made-up data)")
ax_52.set_xlabel("month")
ax_52.set_ylabel("visits per month")
plt.show()


In [ ]:
# Self-check: just run this cell.
self_check("Exercise 5.2", [
    (lambda: not (len(ax_52.patches) > 0 and len(ax_52.get_lines()) == 0), "That is a bar chart. For a line, use ax_52.plot(x, y)."),
    (lambda: len(ax_52.get_lines()) > 0, "No line yet. Use ax_52.plot(ed_visits[\"month\"], ed_visits[\"visits\"])."),
    (lambda: list(ax_52.get_lines()[0].get_ydata()) == ed_visits["visits"].tolist(), "The line should show the visits column on the y axis."),
    (lambda: ax_52.get_title().strip() != "", "Add a title: ax_52.set_title(\"...\")."),
    (lambda: ax_52.get_xlabel().strip() != "", "Add an x-axis label: ax_52.set_xlabel(\"month\")."),
    (lambda: ax_52.get_ylabel().strip() != "", "Add a y-axis label with the unit, such as \"visits per month\"."),
])


In [ ]:
# Exercise 5.3 — fix a misleading chart
systems = ["System A", "System B"]
pct_er = [82, 86]
fig_53, ax_53 = plt.subplots(figsize=(5, 4))
ax_53.bar(systems, pct_er)
ax_53.set_ylim(0, 100)
ax_53.set_ylabel("hospitals with an emergency department (%)")
ax_53.set_title("Share of hospitals with an ED, two systems (made-up data)")
plt.show()


In [ ]:
# Self-check: just run this cell.
self_check("Exercise 5.3", [
    (lambda: len(ax_53.patches) == 2, "Keep the two bars: ax_53.bar(systems, pct_er)."),
    (lambda: ax_53.get_ylim()[0] == 0, lambda: f"The y axis starts at {ax_53.get_ylim()[0]:g}. A bar chart's axis must start at 0: ax_53.set_ylim(0, 100)."),
    (lambda: ax_53.get_ylabel().strip() != "", "Add a y-axis label that names the unit."),
    (lambda: ("%" in ax_53.get_ylabel()) or ("percent" in ax_53.get_ylabel().lower()), "Say the unit in the y-axis label: percent, or %."),
    (lambda: ax_53.get_title().strip() != "", "Add a title that says what is shown."),
])


## Part 6. Ready check (about 20 minutes)
Five short questions, one from each part. Try each one before you scroll up; then scroll up as much as you
like. The last cell prints **READY**, or the parts worth another look. "Not yet" is useful information,
not a grade.

In [ ]:
# Ready 1 (Python)
def star_band(n):
    if n >= 4:
        return "high"
    elif n == 3:
        return "middle"
    else:
        return "low"


In [ ]:
# Self-check: just run this cell.
self_check("Ready 1", [
    (lambda: star_band(5) is not None, "star_band still returns None. Return \"high\", \"middle\" or \"low\"."),
    (lambda: [star_band(n) for n in [5, 4, 3, 2, 1]] == ["high", "high", "middle", "low", "low"],
        lambda: f"For 5, 4, 3, 2, 1 your function returns {[star_band(n) for n in [5, 4, 3, 2, 1]]}. Expected high, high, middle, low, low."),
])


In [ ]:
# Ready 2 (pandas I)
r2_tx_cah = len(hospitals[(hospitals["State"] == "TX") & (hospitals["Hospital Type"] == "Critical Access Hospitals")])
r2_tx_cah


In [ ]:
# Self-check: just run this cell.
self_check("Ready 2", [
    (lambda: r2_tx_cah is not None, NOT_NONE),
    (lambda: r2_tx_cah == ((hospitals["State"] == "TX") & (hospitals["Hospital Type"] == "Critical Access Hospitals")).sum(),
        "Filter on two conditions, (State is TX) & (Hospital Type is Critical Access Hospitals), then count the rows with len()."),
])


In [ ]:
# Ready 3 (pandas II)
r3_rated = hospitals["rating"].dropna()
r3_share_high = (r3_rated >= 4).mean()      # the mean of True/False is the share of True
r3_share_high


In [ ]:
# Self-check: just run this cell.
self_check("Ready 3", [
    (lambda: r3_share_high is not None, NOT_NONE),
    (lambda: 0 <= float(r3_share_high) <= 1, "Give a share between 0 and 1, not a percent or a count."),
    (lambda: not close(r3_share_high, (hospitals["rating"] >= 4).sum() / len(hospitals), 1e-4),
        "You divided by ALL hospitals, including those without a rating. Use only the rated ones: hospitals[\"rating\"].dropna()."),
    (lambda: close(r3_share_high, (hospitals["rating"].dropna() >= 4).mean(), 1e-4), "Not quite. Count rated hospitals with rating >= 4, divided by the number of rated hospitals."),
])


In [ ]:
# Ready 4 (SQL)
r4_top_cah = sql('''
    SELECT "State", COUNT(*) AS n
    FROM hospitals
    WHERE "Hospital Type" = 'Critical Access Hospitals'
    GROUP BY "State"
    ORDER BY n DESC
    LIMIT 5
''')
r4_top_cah


In [ ]:
# Self-check: just run this cell.
self_check("Ready 4", [
    (lambda: r4_top_cah is not None, NOT_NONE),
    (lambda: isinstance(r4_top_cah, pd.DataFrame) and {"State", "n"} <= set(r4_top_cah.columns),
        "Use sql('''...''') and name the columns State and n (COUNT(*) AS n)."),
    (lambda: len(r4_top_cah) == 5, "Return five rows: LIMIT 5."),
    (lambda: r4_top_cah["n"].tolist() == hospitals.loc[hospitals["Hospital Type"] == "Critical Access Hospitals", "State"].value_counts().head(5).tolist(),
        "The counts do not match. Filter to Critical Access Hospitals with WHERE, GROUP BY \"State\", ORDER BY n DESC."),
])


In [ ]:
# Ready 5 (charts)
fig_r5, ax_r5 = plt.subplots(figsize=(6, 3.5))
ax_r5.barh(r4_top_cah["State"], r4_top_cah["n"])
ax_r5.invert_yaxis()                                   # largest at the top
ax_r5.set_title("States with the most Critical Access Hospitals (CMS)")
ax_r5.set_xlabel("Critical Access Hospitals")
plt.show()


In [ ]:
# Self-check: just run this cell.
self_check("Ready 5", [
    (lambda: len(ax_r5.patches) == 5, lambda: f"You have {len(ax_r5.patches)} bars; draw one per state from Ready 4 with ax_r5.barh(...)."),
    (lambda: ax_r5.get_xlim()[0] == 0, "Bars must start at zero. Remove any set_xlim."),
    (lambda: ax_r5.get_title().strip() != "", "Add a title."),
    (lambda: ax_r5.get_xlabel().strip() != "", "Add an x-axis label that names the unit (for a horizontal bar chart the values run along x)."),
])


In [ ]:
# Your result: run this last.
ready_parts = {"Ready 1": "Part 1 (Python basics)", "Ready 2": "Part 2 (pandas I)", "Ready 3": "Part 3 (pandas II)",
               "Ready 4": "Part 4 (SQL)", "Ready 5": "Part 5 (charts)"}
revisit = [part for key, part in ready_parts.items() if not RESULTS.get(key, False)]
open_exercises = [k for k, ok in RESULTS.items() if not ok and not k.startswith("Ready")]

if not revisit:
    print("READY. You have the tools the labs use. See you in week 1.")
else:
    print("Not yet. Revisit: " + "; ".join(revisit) + ".")
    print("That is normal for a first pass. Rerun this cell after you redo those parts.")
if open_exercises:
    print("Exercises still showing 'Not yet': " + ", ".join(open_exercises))


## What comes next
- **Lab 0 — Setup and sanity check.** Ten minutes. You can now read every line of it.
- **Lab 1 — The U.S. healthcare data landscape.** You will profile these same columns, count rated and unrated
  hospitals, and compare groups, in SQL and in pandas side by side. Everything it asks for, you practiced here.

Keep this notebook. When a lab asks for something you half remember, the worked examples here are a quick
place to look it up.